# 03 — Streaming

This notebook demonstrates streaming support in orqest. Instead of waiting for the LLM to finish generating the entire response, you can receive partial results as they arrive — essential for building responsive UIs and real-time API endpoints.

orqest provides three streaming primitives on `BaseAgent`:

| Method | Returns | Use case |
|--------|---------|----------|
| `stream_output()` | Async generator of `OutputT` partials | Stream validated structured output as it builds up |
| `stream_events()` | Async generator of `AgentStreamEvent` | Full visibility — model tokens *and* tool call/result events |
| `call_model_stream()` | `StreamedRunResult` context manager | Full control — access any streaming method on the raw result |

**Prerequisites:**
- A `.env` file in the project root with `LLM_API_KEY` and `LLM_MODEL` set

## 1. Setup

Load config and define a simple agent with structured output.

In [1]:
from pydantic import BaseModel, Field
from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-4.1


In [2]:
class AnalysisOutput(BaseModel):
    """Structured output for our analysis agent."""
    title: str = Field(description="Short title for the analysis")
    summary: str = Field(description="A concise summary")
    key_findings: list[str] = Field(description="Main findings as bullet points")
    sentiment: str = Field(description="Overall sentiment: positive, negative, or neutral")


class AnalysisAgent(BaseAgent[GlobalState, AnalysisOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="analysis_agent",
            system_prompt=(
                "You are an analysis assistant. Analyze the user's text and "
                "provide a structured analysis with title, summary, key findings, "
                "and sentiment."
            ),
            output_type=AnalysisOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> AnalysisOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


agent = AnalysisAgent(model=config.llm_model, api_key=config.llm_api_key)

## 2. Streaming structured output with `stream_output()`

`stream_output()` is an async generator that yields partial `OutputT` instances as the LLM generates tokens. Each yield is a progressively more complete Pydantic model — fields fill in as data arrives.

**Key:** set `debounce_by=0` to see every intermediate partial. The default (`0.1s`) can be too aggressive for fast models, collapsing everything into a single final chunk.

In [3]:
import time

state = GlobalState()
state.add_message(
    "user",
    "The latest quarterly earnings report shows a 15% increase in revenue, "
    "driven primarily by cloud services growth. However, hardware sales declined "
    "by 8% year-over-year. The company announced plans to invest $2B in AI "
    "infrastructure over the next two years."
)

print("Streaming structured output (debounce_by=0)...\n")
chunk_count = 0
t0 = time.time()

async for partial in agent.stream_output(
    state.get_latest_message("user"), state, debounce_by=0
):
    chunk_count += 1
    elapsed = time.time() - t0
    # Show how the output fills in progressively
    print(f"--- Chunk {chunk_count} [{elapsed:.2f}s] ---")
    print(f"  title:         {partial.title!r}")
    summary_preview = partial.summary[:50] + "..." if len(partial.summary) > 50 else partial.summary
    print(f"  summary:       {summary_preview!r}")
    print(f"  key_findings:  {len(partial.key_findings)} items")
    print(f"  sentiment:     {partial.sentiment!r}")
    print()

elapsed = time.time() - t0
print(f"Received {chunk_count} chunks in {elapsed:.2f}s")

Streaming structured output (debounce_by=0)...

--- Chunk 1 [2.05s] ---
  title:         'Quarterly Earnings Report Analysis'
  summary:       "The company's latest quarterly earnings report rev..."
  key_findings:  3 items
  sentiment:     'positive'

--- Chunk 2 [2.07s] ---
  title:         'Quarterly Earnings Report Analysis'
  summary:       "The company's latest quarterly earnings report rev..."
  key_findings:  3 items
  sentiment:     'positive'

Received 2 chunks in 2.07s


Notice how each chunk shows the output filling in over time — early chunks have empty fields (`''`, `0 items`) that progressively populate. The timestamps prove tokens are arriving incrementally, not all at once.

## 3. Raw token streaming with `call_model_stream()`

`call_model_stream()` gives you the raw `StreamedRunResult` from pydantic-ai. You can use `stream_responses()` to see every intermediate `ModelResponse` — this shows the actual token-by-token generation most clearly.

In [4]:
import json

state2 = GlobalState()
state2.add_message(
    "user",
    "Renewable energy adoption has accelerated globally, with solar capacity "
    "doubling in the past three years."
)

print("Raw response streaming (debounce_by=0)...\n")
t0 = time.time()
response_count = 0

async with agent.call_model_stream(state2.get_latest_message("user"), state2) as streamed:
    async for response, is_last in streamed.stream_responses(debounce_by=0):
        response_count += 1
        elapsed = time.time() - t0

        # Extract the tool call args (which is the JSON being built up)
        for part in response.parts:
            if hasattr(part, "args"):
                # Show how the JSON grows token by token
                args_str = part.args if isinstance(part.args, str) else json.dumps(part.args)
                preview = args_str[:80] + "..." if len(args_str) > 80 else args_str
                print(f"[{elapsed:.2f}s] Response #{response_count} ({len(args_str)} chars) {'[FINAL]' if is_last else ''}")
                print(f"  {preview}")
                print()

    # Get the final validated output
    output = await streamed.get_output()

elapsed = time.time() - t0
print(f"Streamed {response_count} responses in {elapsed:.2f}s")
print(f"\nFinal output:")
print(f"  Title: {output.title}")
print(f"  Sentiment: {output.sentiment}")
print(f"\nHistory messages: {len(state2.message_history)}")

Raw response streaming (debounce_by=0)...

[1.25s] Response #1 (25 chars) 
  {"title":"Global Surge in

[1.26s] Response #2 (42 chars) 
  {"title":"Global Surge in Renewable Energy

[1.27s] Response #3 (54 chars) 
  {"title":"Global Surge in Renewable Energy Adoption","

[1.28s] Response #4 (64 chars) 
  {"title":"Global Surge in Renewable Energy Adoption","summary":"

[1.28s] Response #5 (73 chars) 
  {"title":"Global Surge in Renewable Energy Adoption","summary":"Renewable

[1.30s] Response #6 (81 chars) 
  {"title":"Global Surge in Renewable Energy Adoption","summary":"Renewable energy...

[1.30s] Response #7 (100 chars) 
  {"title":"Global Surge in Renewable Energy Adoption","summary":"Renewable energy...

[1.33s] Response #8 (104 chars) 
  {"title":"Global Surge in Renewable Energy Adoption","summary":"Renewable energy...

[1.33s] Response #9 (123 chars) 
  {"title":"Global Surge in Renewable Energy Adoption","summary":"Renewable energy...

[1.36s] Response #10 (140 chars) 
  {"ti

## 4. Multi-turn streaming

Streaming methods manage conversation history the same way `call_model()` does. After the stream is consumed, `state.message_history` is updated. A second streaming call picks up the full history from the first.

In [5]:
state3 = GlobalState()

# Turn 1 — streaming
state3.add_message("user", "Electric vehicle sales grew 40% last year globally.")
last_output = None
chunk_count = 0
async for partial in agent.stream_output(
    state3.get_latest_message("user"), state3, debounce_by=0
):
    last_output = partial
    chunk_count += 1

print(f"Turn 1 — {last_output.title} ({chunk_count} chunks)")
print(f"History after turn 1: {len(state3.message_history)} messages")

# Turn 2 — the agent sees the full conversation from turn 1
state3.add_message("user", "Now analyze the environmental impact of this EV growth.")
last_output = None
chunk_count = 0
async for partial in agent.stream_output(
    state3.get_latest_message("user"), state3, debounce_by=0
):
    last_output = partial
    chunk_count += 1

print(f"\nTurn 2 — {last_output.title} ({chunk_count} chunks)")
print(f"History after turn 2: {len(state3.message_history)} messages")
print(f"\nKey findings from turn 2:")
for finding in last_output.key_findings:
    print(f"  - {finding}")

Turn 1 — Global Electric Vehicle Sales Surge (2 chunks)
History after turn 1: 3 messages

Turn 2 — Environmental Impact of EV Sales Growth (2 chunks)
History after turn 2: 6 messages

Key findings from turn 2:
  - Higher EV adoption generally leads to lower tailpipe emissions, reducing urban air pollution and greenhouse gases compared to traditional vehicles.
  - EVs can help mitigate climate change if charged with electricity from renewable sources; benefits are smaller if grids rely on fossil fuels.
  - Production of EV batteries involves significant energy use and resource extraction (lithium, cobalt, nickel), which may cause environmental harm if not managed sustainably.
  - Greater demand for EVs encourages investments in clean energy infrastructure and battery recycling initiatives.


## 5. Streaming with tool visibility — `stream_events()`

When agents use tools, `stream_output()` and `call_model_stream()` execute tools internally but don't surface those events. `stream_events()` uses pydantic-ai's `Agent.iter()` under the hood, yielding **every** event — model response tokens, tool calls, and tool results.

This is critical for building UIs that show the agent's reasoning process: "Calling weather API..." → "Got result" → "Generating response..."

In [6]:
from pydantic_ai import Tool


# Define a tool the agent can call
def get_market_data(sector: str) -> str:
    """Look up recent market performance data for a sector."""
    data = {
        "technology": "Tech sector up 12% YTD, led by AI and cloud companies.",
        "energy": "Energy sector flat, oil prices stable around $75/barrel.",
        "healthcare": "Healthcare up 5% YTD, biotech M&A activity increasing.",
    }
    return data.get(sector.lower(), f"No data available for {sector}.")


def get_economic_indicator(indicator: str) -> str:
    """Look up a macroeconomic indicator."""
    indicators = {
        "gdp": "GDP growth: 2.8% annualized (Q4).",
        "inflation": "CPI inflation: 3.1% year-over-year.",
        "unemployment": "Unemployment rate: 3.9%.",
    }
    return indicators.get(indicator.lower(), f"Unknown indicator: {indicator}.")


class ResearchOutput(BaseModel):
    """Output from the research agent."""
    report: str = Field(description="A research report incorporating the data")
    sources_used: list[str] = Field(description="Which data sources were consulted")


class ResearchAgent(BaseAgent[GlobalState, ResearchOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="research_agent",
            system_prompt=(
                "You are a market research assistant with access to tools. "
                "Use the available tools to gather data before writing your report. "
                "Always call at least one tool."
            ),
            output_type=ResearchOutput,
            model=model,
            api_key=api_key,
            tools=[Tool(get_market_data), Tool(get_economic_indicator)],
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> ResearchOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


research_agent = ResearchAgent(model=config.llm_model, api_key=config.llm_api_key)

In [7]:
state4 = GlobalState()
state4.add_message("user", "Give me a brief report on the tech sector and current GDP.")

print("Streaming events (model responses + tool calls)...\n")
t0 = time.time()

async for event in research_agent.stream_events(state4.get_latest_message("user"), state4):
    elapsed = time.time() - t0
    kind = event.event_kind

    if kind == "part_start":
        part_type = type(event.part).__name__
        print(f"[{elapsed:.2f}s]  part_start  → {part_type}")
    elif kind == "part_delta":
        delta_type = type(event.delta).__name__
        print(f"[{elapsed:.2f}s]  part_delta  → {delta_type}")
    elif kind == "part_end":
        part_type = type(event.part).__name__
        preview = ""
        if hasattr(event.part, "args"):
            args = event.part.args if isinstance(event.part.args, str) else str(event.part.args)
            preview = f" ({args[:50]}...)" if len(args) > 50 else f" ({args})"
        print(f"[{elapsed:.2f}s]  part_end    → {part_type}{preview}")
    elif kind == "function_tool_call":
        print(f"[{elapsed:.2f}s]  TOOL CALL   → {event.part.tool_name}({event.part.args})")
    elif kind == "function_tool_result":
        content = str(event.result.content)
        preview = content[:60] + "..." if len(content) > 60 else content
        print(f"[{elapsed:.2f}s]  TOOL RESULT → {preview}")
    elif kind == "final_result":
        print(f"[{elapsed:.2f}s]  FINAL RESULT (output schema matched)")
    else:
        print(f"[{elapsed:.2f}s]  {kind}")

elapsed = time.time() - t0
print(f"\nCompleted in {elapsed:.2f}s")
print(f"History messages: {len(state4.message_history)}")

Streaming events (model responses + tool calls)...

[1.06s]  part_start  → ToolCallPart
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_end    → ToolCallPart ({"sector": "technology"})
[1.06s]  part_start  → ToolCallPart
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_delta  → ToolCallPartDelta
[1.06s]  part_delta  → ToolCallPartDelta
[1.07s]  part_end    → ToolCallPart ({"indicator": "GDP"})
[1.07s]  TOOL CALL   → get_market_data({"sector": "technology"})
[1.07s]  TOOL CALL   → get_economic_indicator({"indicator": "GDP"})
[1.07s]  TOOL RESULT → Tech sector up 12% YTD, led by AI and cloud companies.
[1.07s]  TOOL RESULT → GDP growth: 2.8% annualized (Q4).
[1.74s]  part_start  → ToolCallPart
[1.74s]  FINAL RESULT (output schema mat

The event stream shows the full agent lifecycle:

1. **Model generates tool calls** (`part_start` → `part_delta` → `part_end` for `ToolCallPart`)
2. **Tools execute** (`function_tool_call` → `function_tool_result`)
3. **Model generates final output** using tool results (`part_start` → `final_result` → `part_end`)

This is what you'd use to build a UI that shows "Fetching market data..." → "Looking up GDP..." → "Writing report..." in real-time.

## 6. Transport integration — FastAPI SSE

Because `stream_output()` and `stream_events()` are plain async generators, they plug directly into any async transport. Here's how you'd wire them to FastAPI Server-Sent Events endpoints:

```python
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI()
agent = AnalysisAgent(model="openai:gpt-4o", api_key="sk-...")


# Structured output streaming
@app.post("/analyze/stream")
async def analyze_stream(text: str):
    state = GlobalState()
    state.add_message("user", text)

    async def event_generator():
        async for partial in agent.stream_output(
            state.get_latest_message("user"), state, debounce_by=0.05
        ):
            yield f"data: {partial.model_dump_json()}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
    )


# Full event streaming (including tool calls)
@app.post("/research/stream")
async def research_stream(text: str):
    state = GlobalState()
    state.add_message("user", text)

    async def event_generator():
        async for event in research_agent.stream_events(
            state.get_latest_message("user"), state
        ):
            yield f"data: {json.dumps({'kind': event.event_kind})}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
    )
```

**`debounce_by` tuning:** Use `0` for maximum granularity (every validated partial), or a small value like `0.05` to reduce the number of SSE events while still feeling real-time. The default `0.1` is often too aggressive.

The same pattern works with WebSockets, gRPC streams, or any other async transport — orqest stays transport-agnostic.

## Summary

| Method | Level | Returns | Tool visibility | History update |
|--------|-------|---------|-----------------|----------------|
| `call_model()` | High | `AgentRunResult` (complete) | Hidden | After `await` |
| `stream_output()` | High | Async generator of `OutputT` partials | Hidden | After generator exhausted |
| `stream_events()` | High | Async generator of `AgentStreamEvent` | Visible | After generator exhausted |
| `call_model_stream()` | Low | `StreamedRunResult` context manager | Hidden | After context exit |

All four manage `state.message_history` identically — you can freely mix streaming and non-streaming calls in the same conversation.